# 14h -- Robust Tracklet Pooling Sweep

Stage 2 rerun using the 14c TTA recipe with multi-query top-K retention (`k=24`), followed by CPU repooling across robust aggregation modes at the 14e B1 anchor.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re
from pathlib import Path

# Guard: detect GPU before importing torch. Kaggle's newest PyTorch builds may
# not support P100 sm_60, so pin the cu124 build requested for this experiment.
if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0 and result.stdout.strip():
        gpu_name, compute_cap = result.stdout.strip().split(",", 1)
        match = re.search(r"(\d+)\.(\d+)", compute_cap)
        if match:
            major, minor = match.groups()
            sm = int(major) * 10 + int(minor)
            if sm < 70:
                print(f"GPU {gpu_name.strip()} sm_{sm}: installing torch 2.4.1+cu124")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1+cu124", "torchvision==0.19.1+cu124",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}: {torch.cuda.get_device_name(index)} ({props.total_memory / 1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Clone Repo And Install Dependencies

In [ ]:
from pathlib import Path

import os, sys, subprocess, shutil, json, time, tarfile, re

REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present; pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))


LOCAL_14H_FILES = {
    "src/stage2_features/robust_pool.py": "\"\"\"Robust aggregation for per-tracklet multi-query embeddings.\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Callable, Iterable\n\nimport numpy as np\n\n\nDEFAULT_MIN_K = 8\nGEOMETRIC_MEDIAN_MAX_ITER = 20\nGEOMETRIC_MEDIAN_EPS = 1e-6\n\n\ndef l2_normalize_vector(vector: np.ndarray, eps: float = 1e-8) -> np.ndarray:\n    \"\"\"Return a float32 L2-normalized copy of one vector.\"\"\"\n    array = np.asarray(vector, dtype=np.float32)\n    norm = float(np.linalg.norm(array))\n    if norm <= eps:\n        return np.zeros_like(array, dtype=np.float32)\n    return (array / norm).astype(np.float32)\n\n\ndef l2_normalize_rows(embeddings: np.ndarray, eps: float = 1e-8) -> np.ndarray:\n    \"\"\"Return row-wise L2-normalized embeddings.\"\"\"\n    array = np.asarray(embeddings, dtype=np.float32)\n    if array.ndim != 2:\n        raise ValueError(f\"Expected a 2D embedding block, got shape {array.shape}\")\n    norms = np.linalg.norm(array, axis=1, keepdims=True)\n    return (array / np.maximum(norms, eps)).astype(np.float32)\n\n\ndef trim_stage2_padding(embeddings: np.ndarray, atol: float = 1e-6) -> np.ndarray:\n    \"\"\"Remove the repeated pooled suffix used to pad short multi-query tracks.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    if rows.shape[0] <= 1:\n        return rows\n\n    last = rows[-1]\n    suffix_start = rows.shape[0] - 1\n    while suffix_start > 0 and np.allclose(rows[suffix_start - 1], last, atol=atol):\n        suffix_start -= 1\n\n    suffix_count = rows.shape[0] - suffix_start\n    if suffix_count <= 1:\n        return rows\n    return rows[:suffix_start]\n\n\ndef softmax_quality_mean(\n    embeddings: np.ndarray,\n    qualities: np.ndarray | None = None,\n    temperature: float = 3.0,\n) -> np.ndarray:\n    \"\"\"Quality-weighted fallback matching Stage 2's softmax pooling shape.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    if qualities is None:\n        return mean_pool(rows)\n\n    quality_array = np.asarray(qualities, dtype=np.float32)\n    if quality_array.ndim != 1 or quality_array.shape[0] != rows.shape[0]:\n        raise ValueError(\n            f\"qualities must have shape ({rows.shape[0]},), got {quality_array.shape}\"\n        )\n    logits = quality_array * float(temperature)\n    logits = logits - np.max(logits)\n    weights = np.exp(logits).astype(np.float32)\n    weights = weights / max(float(weights.sum()), 1e-8)\n    return l2_normalize_vector((rows * weights[:, np.newaxis]).sum(axis=0))\n\n\ndef mean_pool(embeddings: np.ndarray) -> np.ndarray:\n    \"\"\"Arithmetic mean followed by L2 renormalization.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    return l2_normalize_vector(rows.mean(axis=0))\n\n\ndef median_pool(embeddings: np.ndarray) -> np.ndarray:\n    \"\"\"Per-dimension median followed by L2 renormalization.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    return l2_normalize_vector(np.median(rows, axis=0))\n\n\ndef geometric_median_pool(\n    embeddings: np.ndarray,\n    max_iter: int = GEOMETRIC_MEDIAN_MAX_ITER,\n    eps: float = GEOMETRIC_MEDIAN_EPS,\n) -> np.ndarray:\n    \"\"\"Weiszfeld geometric median followed by L2 renormalization.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    estimate = rows.mean(axis=0)\n\n    for _ in range(max_iter):\n        distances = np.linalg.norm(rows - estimate[np.newaxis, :], axis=1)\n        exact = distances < eps\n        if np.any(exact):\n            estimate = rows[int(np.argmax(exact))]\n            break\n\n        weights = 1.0 / np.maximum(distances, eps)\n        next_estimate = (rows * weights[:, np.newaxis]).sum(axis=0) / weights.sum()\n        if float(np.linalg.norm(next_estimate - estimate)) < eps:\n            estimate = next_estimate\n            break\n        estimate = next_estimate\n\n    return l2_normalize_vector(estimate)\n\n\ndef medoid_pool(embeddings: np.ndarray) -> np.ndarray:\n    \"\"\"Return the input row with highest summed cosine similarity to all rows.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    similarity = rows @ rows.T\n    medoid_index = int(np.argmax(similarity.sum(axis=1)))\n    return rows[medoid_index].astype(np.float32)\n\n\ndef _top_by_similarity_to_reference(\n    rows: np.ndarray,\n    reference: np.ndarray,\n    keep_count: int,\n) -> np.ndarray:\n    keep = max(1, min(int(keep_count), rows.shape[0]))\n    scores = rows @ l2_normalize_vector(reference)\n    indices = np.argsort(-scores)[:keep]\n    return rows[indices]\n\n\ndef trimmed_mean_pool(embeddings: np.ndarray, trim_fraction: float) -> np.ndarray:\n    \"\"\"Drop the least central rows by cosine-to-mean, then average.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    trim_count = int(np.floor(rows.shape[0] * float(trim_fraction)))\n    keep_count = max(1, rows.shape[0] - trim_count)\n    reference = mean_pool(rows)\n    kept = _top_by_similarity_to_reference(rows, reference, keep_count)\n    return mean_pool(kept)\n\n\ndef top_to_mean_pool(embeddings: np.ndarray, keep_count: int = 12) -> np.ndarray:\n    \"\"\"Average the rows nearest to the arithmetic-mean direction.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    kept = _top_by_similarity_to_reference(rows, mean_pool(rows), keep_count)\n    return mean_pool(kept)\n\n\ndef top_to_medoid_pool(embeddings: np.ndarray, keep_count: int = 12) -> np.ndarray:\n    \"\"\"Average the rows nearest to the medoid direction.\"\"\"\n    rows = l2_normalize_rows(embeddings)\n    kept = _top_by_similarity_to_reference(rows, medoid_pool(rows), keep_count)\n    return mean_pool(kept)\n\n\nAGGREGATION_MODES: dict[str, Callable[[np.ndarray], np.ndarray]] = {\n    \"mean\": mean_pool,\n    \"median\": median_pool,\n    \"geo_median\": geometric_median_pool,\n    \"geometric_median\": geometric_median_pool,\n    \"medoid\": medoid_pool,\n    \"trimmed_mean_10\": lambda rows: trimmed_mean_pool(rows, 0.10),\n    \"trimmed_mean_25\": lambda rows: trimmed_mean_pool(rows, 0.25),\n    \"top12_to_mean\": lambda rows: top_to_mean_pool(rows, 12),\n    \"top12_to_medoid\": lambda rows: top_to_medoid_pool(rows, 12),\n}\n\nSWEEP_MODES = (\n    \"mean\",\n    \"median\",\n    \"geo_median\",\n    \"medoid\",\n    \"trimmed_mean_10\",\n    \"trimmed_mean_25\",\n    \"top12_to_mean\",\n    \"top12_to_medoid\",\n)\n\n\ndef aggregate_tracklet_embeddings(\n    embeddings: np.ndarray,\n    mode: str,\n    fallback_embedding: np.ndarray | None = None,\n    min_k: int = DEFAULT_MIN_K,\n    trim_padding: bool = True,\n) -> tuple[np.ndarray, bool]:\n    \"\"\"Aggregate one tracklet's multi-query rows.\n\n    Returns ``(embedding, used_fallback)``. If fewer than ``min_k`` rows are\n    available, the provided fallback embedding is returned, or a mean pool when\n    no fallback is available.\n    \"\"\"\n    rows = trim_stage2_padding(embeddings) if trim_padding else l2_normalize_rows(embeddings)\n    if rows.shape[0] == 0:\n        if fallback_embedding is None:\n            raise ValueError(\"Cannot aggregate an empty embedding block without fallback\")\n        return l2_normalize_vector(fallback_embedding), True\n\n    if rows.shape[0] < int(min_k):\n        if fallback_embedding is not None:\n            return l2_normalize_vector(fallback_embedding), True\n        return mean_pool(rows), True\n\n    try:\n        aggregator = AGGREGATION_MODES[mode]\n    except KeyError as exc:\n        choices = \", \".join(sorted(AGGREGATION_MODES))\n        raise ValueError(f\"Unknown robust pooling mode '{mode}'. Choices: {choices}\") from exc\n\n    return aggregator(rows), False\n\n\ndef aggregate_embedding_matrix(\n    multi_query_embeddings: np.ndarray,\n    mode: str,\n    fallback_embeddings: np.ndarray | None = None,\n    min_k: int = DEFAULT_MIN_K,\n    trim_padding: bool = True,\n) -> tuple[np.ndarray, int]:\n    \"\"\"Aggregate an ``(N, K, D)`` multi-query tensor into ``(N, D)``.\"\"\"\n    tensor = np.asarray(multi_query_embeddings, dtype=np.float32)\n    if tensor.ndim != 3:\n        raise ValueError(f\"Expected multi-query tensor shape (N, K, D), got {tensor.shape}\")\n    if fallback_embeddings is not None:\n        fallback_embeddings = np.asarray(fallback_embeddings, dtype=np.float32)\n        if fallback_embeddings.shape != (tensor.shape[0], tensor.shape[2]):\n            raise ValueError(\n                \"fallback_embeddings must have shape \"\n                f\"{(tensor.shape[0], tensor.shape[2])}, got {fallback_embeddings.shape}\"\n            )\n\n    pooled: list[np.ndarray] = []\n    fallback_count = 0\n    for index, rows in enumerate(tensor):\n        fallback = None if fallback_embeddings is None else fallback_embeddings[index]\n        embedding, used_fallback = aggregate_tracklet_embeddings(\n            rows,\n            mode=mode,\n            fallback_embedding=fallback,\n            min_k=min_k,\n            trim_padding=trim_padding,\n        )\n        pooled.append(embedding)\n        fallback_count += int(used_fallback)\n\n    return np.stack(pooled, axis=0).astype(np.float32), fallback_count\n\n\ndef available_modes() -> Iterable[str]:\n    \"\"\"Return the public sweep modes in execution order.\"\"\"\n    return SWEEP_MODES",
    "scripts/repool_stage2.py": "\"\"\"Re-pool Stage 2 multi-query embeddings with robust aggregators.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom src.stage2_features.robust_pool import (\n    DEFAULT_MIN_K,\n    aggregate_embedding_matrix,\n    available_modes,\n)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--stage2-dir\", type=Path, required=True)\n    parser.add_argument(\"--mode\", choices=tuple(available_modes()), required=True)\n    parser.add_argument(\"--min-k\", type=int, default=DEFAULT_MIN_K)\n    parser.add_argument(\"--output\", type=Path, default=None)\n    parser.add_argument(\"--summary\", type=Path, default=None)\n    parser.add_argument(\"--no-trim-padding\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef repool_stage2(\n    stage2_dir: Path,\n    mode: str,\n    min_k: int = DEFAULT_MIN_K,\n    output: Path | None = None,\n    summary: Path | None = None,\n    trim_padding: bool = True,\n) -> dict:\n    stage2_dir = Path(stage2_dir)\n    mq_path = stage2_dir / \"multi_query_embeddings.npz\"\n    fallback_path = stage2_dir / \"embeddings.npy\"\n    if not mq_path.exists():\n        raise FileNotFoundError(mq_path)\n    if not fallback_path.exists():\n        raise FileNotFoundError(fallback_path)\n\n    with np.load(mq_path) as data:\n        if \"embeddings\" not in data:\n            raise KeyError(f\"{mq_path} does not contain an 'embeddings' array\")\n        multi_query_embeddings = data[\"embeddings\"].astype(np.float32)\n\n    fallback_embeddings = np.load(fallback_path).astype(np.float32)\n    pooled, fallback_count = aggregate_embedding_matrix(\n        multi_query_embeddings,\n        mode=mode,\n        fallback_embeddings=fallback_embeddings,\n        min_k=min_k,\n        trim_padding=trim_padding,\n    )\n\n    output = output or (stage2_dir / f\"embeddings_{mode}.npy\")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    np.save(output, pooled.astype(np.float32))\n\n    norms = np.linalg.norm(pooled, axis=1)\n    payload = {\n        \"mode\": mode,\n        \"min_k\": int(min_k),\n        \"stage2_dir\": str(stage2_dir),\n        \"multi_query_path\": str(mq_path),\n        \"fallback_path\": str(fallback_path),\n        \"output\": str(output),\n        \"input_shape\": list(multi_query_embeddings.shape),\n        \"output_shape\": list(pooled.shape),\n        \"fallback_count\": int(fallback_count),\n        \"trim_padding\": bool(trim_padding),\n        \"norm_min\": float(norms.min()) if norms.size else None,\n        \"norm_max\": float(norms.max()) if norms.size else None,\n    }\n\n    if summary is not None:\n        summary.parent.mkdir(parents=True, exist_ok=True)\n        summary.write_text(json.dumps(payload, indent=2), encoding=\"utf-8\")\n\n    return payload\n\n\ndef main() -> None:\n    args = parse_args()\n    payload = repool_stage2(\n        stage2_dir=args.stage2_dir,\n        mode=args.mode,\n        min_k=args.min_k,\n        output=args.output,\n        summary=args.summary,\n        trim_padding=not args.no_trim_padding,\n    )\n    print(json.dumps(payload, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()",
}
for rel_path, content in LOCAL_14H_FILES.items():
    target_path = PROJECT / rel_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_text(content, encoding="utf-8")
print("Injected 14h robust pooling sources into cloned repo")

print(f"Repo ready at {PROJECT}")

In [ ]:
import subprocess, sys

import importlib

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Keep torch/torchvision pinned above; install project dependencies around it.
pip("--no-deps", "ultralytics")
pip("filterpy", "ftfy", "lapx")
pip("--no-deps", "boxmot==11.0.3")

try:
    import torchreid
    print("torchreid ok")
except ImportError:
    pip("git+https://github.com/KaiyangZhou/deep-person-reid.git")

try:
    import faiss
    print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip("faiss-gpu")
    except Exception:
        pip("faiss-cpu")

pip("timm", "motmetrics")
pip("loguru", "omegaconf", "rich", "networkx>=3.1", "click")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))
print("Dependencies installed")

In [ ]:
FAILED = []
for label, module in [
    ("ultralytics", "ultralytics"),
    ("boxmot", "boxmot"),
    ("torch", "torch"),
    ("torchreid", "torchreid"),
    ("timm", "timm"),
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("cv2", "cv2"),
    ("omegaconf", "omegaconf"),
]:
    try:
        __import__(module)
        print(f"  OK {label}")
    except ImportError as exc:
        print(f"  MISSING {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED}")
print("All required modules importable")

## 2. Mount MTMC Weights

In [ ]:
import shutil

import json, os
from pathlib import Path

import zipfile

weights_candidates = [
    Path("/kaggle/input/mtmc-weights"),
    Path("/kaggle/input/yahiaakhalafallah-mtmc-weights"),
    Path("/kaggle/input/yahiaakhalafallah/mtmc-weights"),
    Path("/kaggle/input/gumfreddy-mtmc-weights"),
    Path("/kaggle/input/gumfreddy/mtmc-weights"),
]
WEIGHTS_INPUT = next((path for path in weights_candidates if path.exists()), None)
if WEIGHTS_INPUT is None:
    candidates = [
        path
        for path in Path("/kaggle/input").rglob("*")
        if path.is_dir() and "mtmc" in path.name.lower() and "weight" in path.name.lower()
    ]
    if candidates:
        WEIGHTS_INPUT = candidates[0]
    else:
        tried = ", ".join(str(path) for path in weights_candidates)
        raise FileNotFoundError(
            f"MTMC weights dataset not found; attach yahiaakhalafallah/mtmc-weights in kernel-metadata.json. Tried: {tried}"
        )
print(f"MTMC weights input: {WEIGHTS_INPUT}")

MODELS_DST = PROJECT / "models"
if MODELS_DST.is_symlink():
    MODELS_DST.unlink()
if MODELS_DST.exists():
    shutil.rmtree(MODELS_DST)
print(f"Copying models from {WEIGHTS_INPUT} ...")
shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

for zip_path in sorted(MODELS_DST.rglob("*.zip")):
    print(f"Extracting {zip_path.relative_to(MODELS_DST)}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(zip_path.parent)
    zip_path.unlink()

for subdir in ["detection", "reid", "tracker"]:
    (MODELS_DST / subdir).mkdir(exist_ok=True)
for path in list(MODELS_DST.glob("*.pt")):
    if "yolo" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "detection" / path.name))
    elif "osnet" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "tracker" / path.name))
for path in list(MODELS_DST.glob("*.pth")):
    shutil.move(str(path), str(MODELS_DST / "reid" / path.name))
for path in list(MODELS_DST.glob("*.pkl")):
    shutil.move(str(path), str(MODELS_DST / "reid" / path.name))
for path in list(MODELS_DST.glob("*.json")):
    if path.name != "dataset-metadata.json":
        shutil.move(str(path), str(MODELS_DST / "reid" / path.name))

essential = [
    PROJECT / "models" / "reid" / "transreid_cityflowv2_best.pth",
]
missing = [str(path) for path in essential if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing essential weights: {missing}")
print("Baseline ReID weights ready")

In [ ]:
import json, os
from pathlib import Path

# 14h v1 TTA patch: use the proven 14c recipe (primary 4-view, DINOv2 tertiary 2-view).
os.environ["MTMC_TTA_RECIPE"] = "14c_v1"

TTA_RECIPE = {
    "variant": "14h v1: 14c TTA + robust tracklet pooling",
    "primary_views": ["original", "hflip", "scale_0.95", "scale_1.05"],
    "tertiary_dinov2_views": ["original", "hflip"],
    "aggregation": "mean_l2",
    "multi_query_k": 24,
}

REID_MODEL_PATH = PROJECT / "src" / "stage2_features" / "reid_model.py"
reid_text = REID_MODEL_PATH.read_text(encoding="utf-8")


def replace_once(text: str, old: str, new: str, label: str) -> str:
    if old not in text:
        raise RuntimeError(f"14h TTA patch anchor not found: {label}")
    return text.replace(old, new, 1)

if "MTMC_TTA_RECIPE" not in reid_text:
    reid_text = replace_once(
        reid_text,
        "from typing import List, Optional, Tuple, TYPE_CHECKING\n\nimport cv2",
        "from typing import List, Optional, Tuple, TYPE_CHECKING\n\nimport os\n\nimport cv2",
        "import os",
    )
    reid_text = replace_once(
        reid_text,
        "        self.vit_model = vit_model\n\n        # Auto-detect CLIP normalization",
        """        self.vit_model = vit_model
        self.tta_views: list[str] = []
        tta_recipe = os.environ.get(\"MTMC_TTA_RECIPE\", \"\")
        if tta_recipe in (\"14c_v1\", \"14g_v1\") and self.is_transreid:
            if \"dinov2\" in vit_model.lower():
                if tta_recipe == \"14g_v1\":
                    self.tta_views = [\"original\", \"hflip\", \"scale_0.95\", \"scale_1.05\"]
                else:
                    self.tta_views = [\"original\", \"hflip\"]
            else:
                self.tta_views = [\"original\", \"hflip\", \"scale_0.95\", \"scale_1.05\"]
            self.normalize_views = True

        # Auto-detect CLIP normalization""",
        "init tta views",
    )
    reid_text = replace_once(
        reid_text,
        "            + (\", normalize_views=true\" if self.normalize_views else \"\")\n        )",
        "            + (\", normalize_views=true\" if self.normalize_views else \"\")\n            + (f\", tta_views={self.tta_views}\" if self.tta_views else \"\")\n        )",
        "log tta views",
    )
    reid_text = replace_once(
        reid_text,
        "    @torch.no_grad()\n    def _extract_batch",
        """    def _apply_tta_view(self, crops: List[np.ndarray], view_name: str) -> List[np.ndarray]:
        if view_name == \"original\":
            return crops
        if view_name == \"hflip\":
            return [cv2.flip(crop, 1) for crop in crops]
        if view_name.startswith(\"scale_\"):
            return self._center_crop_at_scale(crops, float(view_name.split(\"_\", 1)[1]))
        raise ValueError(f\"Unsupported TTA view: {view_name}\")

    @torch.no_grad()
    def _extract_batch""",
        "apply_tta_view helper",
    )
    reid_text = replace_once(
        reid_text,
        "        cam_tensor = self._make_cam_tensor(len(batch_crops), cam_id)\n        views = [self._forward_crops(batch_crops, cam_tensor)]",
        """        cam_tensor = self._make_cam_tensor(len(batch_crops), cam_id)
        if self.tta_views:
            views = [
                self._forward_crops(self._apply_tta_view(batch_crops, view_name), cam_tensor)
                for view_name in self.tta_views
            ]
            views = [self._l2_normalize_rows(view) for view in views]
            pooled = np.mean(np.stack(views, axis=0), axis=0)
            return self._l2_normalize_rows(pooled).astype(np.float32)

        views = [self._forward_crops(batch_crops, cam_tensor)]""",
        "extract_batch exact views",
    )
    REID_MODEL_PATH.write_text(reid_text, encoding="utf-8")

print("Applied 14h TTA patch")
print(json.dumps(TTA_RECIPE, indent=2))

## 3. Mount CityFlowV2 Videos

In [ ]:
import shutil
from pathlib import Path

import re as regex

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount:16s} {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")

candidate_mounts = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = next((path for path in candidate_mounts if path.exists()), None)
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists():
        shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"
            flat_dir = DATA_RAW / flat_name
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

cam_pattern = regex.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(path.name for path in DATA_RAW.iterdir() if path.is_dir() and cam_pattern.match(path.name))
print(f"CityFlowV2 ready: {len(cams)} cameras")
for cam in cams:
    print(f"  {cam}")

## 4. Load 10a Stage 0/1 Checkpoint

In [ ]:
import json, shutil, subprocess, tarfile

import json, os, subprocess, time
import numpy as np

from pathlib import Path
import subprocess

PREV_OWNER_SLUG = "yahiaakhalafallah/mtmc-10a-stages-0-2"
PREV_SLUG = PREV_OWNER_SLUG.split("/", 1)[1]
INPUT_ROOT = Path("/kaggle/input")


def find_input_dir(slug: str, owner_slug: str, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct

    owner, _, kernel = owner_slug.partition("/")
    nested = INPUT_ROOT / "notebooks" / owner / kernel
    if nested.exists():
        return nested

    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    direct_children = list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for path in direct_children:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(hint in name for hint in lowered_hints):
            return path

    for path in INPUT_ROOT.rglob("checkpoint.tar.gz") if INPUT_ROOT.exists() else []:
        parent_text = str(path.parent).lower()
        if lowered_slug in parent_text or all(hint in parent_text for hint in lowered_hints):
            return path.parent

    return direct


def find_checkpoint() -> Path:
    prev_input = find_input_dir(PREV_SLUG, PREV_OWNER_SLUG, hints=("10a", "stages", "0", "2"))
    checkpoint_path = prev_input / "checkpoint.tar.gz"
    if checkpoint_path.exists():
        print(f"10a input: {prev_input}")
        return checkpoint_path

    print(f"checkpoint.tar.gz not found at {checkpoint_path}; trying Kaggle API fallback")
    dl_dir = Path("/tmp/kaggle_10a_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    for candidate in [PREV_OWNER_SLUG, "gumfreddy/mtmc-10a-stages-0-2"]:
        result = subprocess.run(
            [
                "kaggle",
                "kernels",
                "output",
                candidate,
                "--file-pattern",
                r"^checkpoint\.tar\.gz$",
                "-p",
                str(dl_dir),
            ],
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        print(result.stderr)
        checkpoint_path = dl_dir / "checkpoint.tar.gz"
        if checkpoint_path.exists() and checkpoint_path.stat().st_size > 0:
            print(f"10a checkpoint downloaded from {candidate}")
            return checkpoint_path

    searched = [str(path) for path in INPUT_ROOT.rglob("checkpoint.tar.gz")] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f"10a checkpoint.tar.gz not found for {PREV_OWNER_SLUG}; searched mount candidates and API fallback. "
        f"Visible checkpoints under /kaggle/input: {searched[:20]}"
    )


checkpoint = find_checkpoint()

EXTRACT_DIR = Path("/tmp/10a_checkpoint")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)")
with tarfile.open(str(checkpoint), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as handle:
    previous_meta = json.load(handle)
PREV_RUN_NAME = previous_meta["run_name"]
PREV_RUN_DIR = EXTRACT_DIR / PREV_RUN_NAME
print(f"Loaded 10a run: {PREV_RUN_NAME}")
for stage in ["stage1", "stage2"]:
    stage_dir = PREV_RUN_DIR / stage
    print(f"  {stage}: exists={stage_dir.exists()} files={len([p for p in stage_dir.rglob('*') if p.is_file()]) if stage_dir.exists() else 0}")

## 5. Run 14c TTA Stage 2 With Multi-Query K=24

In [ ]:
import shutil

import json, time
from dataclasses import replace
from pathlib import Path
import numpy as np

from datetime import datetime

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)
RUN_NAME = f"run_14h_robust_pooling_v1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREV_RUN_DIR / "stage1", RUN_DIR / "stage1")
print(f"Run  : {RUN_NAME}")
print(f"Stage1 copied from {PREV_RUN_DIR / 'stage1'}")

In [ ]:
import json, os, subprocess, time
from pathlib import Path
import numpy as np

os.chdir(str(PROJECT))

from src.core.config import load_config, save_config
from src.core.io_utils import load_tracklets_by_camera
from src.core.logging_utils import setup_logging
from src.stage2_features import run_stage2

BASELINE_WEIGHTS = "models/reid/transreid_cityflowv2_best.pth"
TERTIARY_SLUG = "09s-dinov2-large-cityflowv2"
TERTIARY_OWNER_SLUG = f"yahiaakhalafallah/{TERTIARY_SLUG}"
TERTIARY_FILENAME = "vehicle_transreid_dinov2_large_cityflowv2_final.pth"


def find_tertiary_weights() -> str:
    candidates = [
        Path("/kaggle/input") / TERTIARY_SLUG / TERTIARY_FILENAME,
        Path("/kaggle/input/notebooks/yahiaakhalafallah") / TERTIARY_SLUG / TERTIARY_FILENAME,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for candidate in input_root.rglob(TERTIARY_FILENAME):
            if TERTIARY_SLUG in str(candidate) or "dinov2" in str(candidate).lower():
                return str(candidate)

    dl_dir = Path("/tmp/kaggle_09s_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        [
            "kaggle",
            "kernels",
            "output",
            TERTIARY_OWNER_SLUG,
            "--file-pattern",
            f"^{TERTIARY_FILENAME}$",
            "-p",
            str(dl_dir),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    downloaded = dl_dir / TERTIARY_FILENAME
    if downloaded.exists() and downloaded.stat().st_size > 0:
        return str(downloaded)

    return str(candidates[0])


TERTIARY_WEIGHTS = find_tertiary_weights()
print(f"Primary TransReID: {BASELINE_WEIGHTS} exists={Path(BASELINE_WEIGHTS).exists()}")
print(f"DINOv2 tertiary : {TERTIARY_WEIGHTS} exists={Path(TERTIARY_WEIGHTS).exists()}")

overrides = [
    f"project.run_name={RUN_NAME}",
    f"project.output_dir={DATA_OUT}",
    f"stage0.input_dir={DATA_RAW}",
    "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
    "stage2.crop.samples_per_tracklet=48",
    "stage2.crop.foreground_masking.enabled=false",
    "stage2.reid.normalize_views=true",
    "stage2.pca.n_components=384",
    "stage2.reid.color_augment=false",
    "stage2.reid.vehicle.concat_patch=false",
    "stage2.reid.vehicle.input_size=[256,256]",
    "stage2.reid.vehicle2.enabled=false",
    "stage2.multi_query.k=24",
    "stage2.camera_bn.enabled=false",
    "stage2.camera_tta.enabled=false",
    "stage2.power_norm.alpha=0.0",
]

if Path(TERTIARY_WEIGHTS).exists():
    overrides += [
        "stage2.reid.vehicle3.enabled=true",
        f"stage2.reid.vehicle3.weights_path={TERTIARY_WEIGHTS}",
        "stage2.reid.vehicle3.input_size=[252,252]",
        "stage2.reid.vehicle3.embedding_dim=1024",
        "stage2.reid.vehicle3.model_name=transreid",
        "stage2.reid.vehicle3.vit_model=vit_large_patch14_dinov2.lvd142m",
        "stage2.reid.vehicle3.clip_normalization=false",
        "stage2.reid.vehicle3.num_cameras=59",
        "stage2.reid.vehicle3.save_separate=true",
    ]
else:
    print("WARNING: DINOv2 tertiary checkpoint not found; vehicle3 will stay disabled")

cfg = load_config("configs/default.yaml", dataset_config="configs/datasets/cityflowv2.yaml", overrides=overrides)
setup_logging(level=cfg.project.get("log_level", "INFO"), log_file=RUN_DIR / "pipeline.log")
save_config(cfg, RUN_DIR / "config.yaml")
video_paths = {}
for cam_dir in sorted(DATA_RAW.glob("S*_c*")):
    if not cam_dir.is_dir():
        continue
    cam_id = cam_dir.name
    video_path = cam_dir / "vdo.avi"
    if video_path.exists():
        video_paths[cam_id] = str(video_path)
tracklets_by_camera = load_tracklets_by_camera(RUN_DIR / "stage1")
print(f"Tracklets: {sum(len(v) for v in tracklets_by_camera.values())} across {len(tracklets_by_camera)} cameras")
print(f"Videos   : {len(video_paths)} discovered")
for camera_id in sorted(tracklets_by_camera):
    print(f"  {camera_id}: tracklets={len(tracklets_by_camera[camera_id])} video={camera_id in video_paths}")

start = time.time()
features = run_stage2(
    cfg,
    tracklets_by_camera,
    video_paths,
    output_dir=RUN_DIR / "stage2",
    smoke_test=False,
    stage0_dir=None,
)
elapsed = time.time() - start
print(f"TTA Stage 2 complete: {len(features)} features in {elapsed / 60:.1f} min")
MULTI_QUERY_PATH = RUN_DIR / "stage2" / "multi_query_embeddings.npz"
if not MULTI_QUERY_PATH.exists():
    raise FileNotFoundError(f"Expected multi-query artifact missing: {MULTI_QUERY_PATH}")
with np.load(MULTI_QUERY_PATH) as mq_data:
    mq_shape = list(mq_data["embeddings"].shape)
print(f"Multi-query artifact: {MULTI_QUERY_PATH} shape={mq_shape}")

## 6. Run Robust Pooling Sweep

In [ ]:
import json, time
from pathlib import Path

from dataclasses import replace

import numpy as np

from scripts.repool_stage2 import repool_stage2
from src.stage3_indexing import run_stage3
from src.stage4_association import run_stage4
from src.stage5_evaluation import run_stage5

AGGREGATION_MODES = [
    "mean",
    "median",
    "geo_median",
    "medoid",
    "trimmed_mean_10",
    "trimmed_mean_25",
    "top12_to_mean",
    "top12_to_medoid",
]

ANCHOR_CONFIG = {
    "aqe_k": 2,
    "fic_regularisation": 0.5,
    "similarity_threshold": 0.48,
    "w_tertiary": 0.525,
}
SWEEP_CONFIGS = [
    {"label": "M0", "block": "drift", "aggregation_mode": "existing_softmax_quality_mean", "notes": "14e B1 anchor drift check on existing Stage-2 embeddings", **ANCHOR_CONFIG},
] + [
    {"label": f"M{index}", "block": "robust_pool", "aggregation_mode": mode, "notes": f"Robust pooling mode: {mode}", **ANCHOR_CONFIG}
    for index, mode in enumerate(AGGREGATION_MODES, start=1)
]

M0_REPRO_TARGET = 0.77936
M0_REPRO_TOL = 0.005
WIN_THRESHOLD = 0.7810
MARGINAL_MIN = 0.7795
NEUTRAL_MIN = 0.7785
MIN_K = 8
SOLVER = "cc"
ALGORITHM = "conflict_free_cc"
LOUVAIN_RES = 0.70
APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)
BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38
INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30
MULTI_QUERY_WEIGHT = 0.00
CAMERA_BIAS = False
CAMERA_BIAS_ITERS = 2
ZONE_MODEL = False
ZONE_BONUS = 0.06
ZONE_PENALTY = 0.04
HIERARCHICAL = False
HIER_CENTROID_TH = 0.45
HIER_MERGE_TH = 0.45
HIER_ORPHAN_TH = 0.40
RERANKING = False
RERANKING_K1 = 20
RERANKING_K2 = 6
RERANKING_LAMBDA = 0.5
CAMERA_PAIR_NORM = False
AFLINK_ENABLED = False
AFLINK_MAX_TIME_GAP_FRAMES = 150
AFLINK_MAX_SPATIAL_GAP_PX = 150.0
AFLINK_MIN_DIRECTION_COS = 0.85
AFLINK_MIN_VELOCITY_RATIO = 0.5
AFLINK_VELOCITY_WINDOW = 5
MTMC_ONLY = False

PRIMARY_EMBEDDINGS_PATH = RUN_DIR / "stage2" / "embeddings.npy"
TERTIARY_DINOV2_PATH = RUN_DIR / "stage2" / "embeddings_tertiary.npy"
MULTI_QUERY_PATH = RUN_DIR / "stage2" / "multi_query_embeddings.npz"
for required_path in [PRIMARY_EMBEDDINGS_PATH, TERTIARY_DINOV2_PATH, MULTI_QUERY_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

ORIGINAL_FEATURES = list(features)
ORIGINAL_PRIMARY_EMBEDDINGS = np.load(PRIMARY_EMBEDDINGS_PATH).astype(np.float32)

GT_DIR = DATA_RAW
expected_cams = ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]
if not any((GT_DIR / cam / "gt" / "gt.txt").exists() for cam in expected_cams):
    dataset_gt = Path("/kaggle/input/data-aicity-2023-track-2")
    extracted_gt = EXTRACT_DIR / "gt_annotations"
    if dataset_gt.exists():
        GT_DIR = dataset_gt
    elif extracted_gt.exists():
        GT_DIR = extracted_gt
    else:
        raise FileNotFoundError("CityFlowV2 ground-truth annotations not found")


def load_metrics(report_path: Path) -> dict:
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    details = payload.get("details", {}) or {}
    error_analysis = details.get("error_analysis", {}) or {}
    return {
        "mtmc_idf1": payload.get("mtmc_idf1", details.get("mtmc_idf1", payload.get("idf1"))),
        "trackeval_idf1": payload.get("idf1"),
        "mota": payload.get("mota"),
        "hota": payload.get("hota"),
        "id_switches": payload.get("id_switches"),
        "conflations": error_analysis.get("conflated_pred"),
        "fragmentations": error_analysis.get("fragmented_gt"),
        "num_pred_ids": payload.get("num_pred_ids", error_analysis.get("total_pred")),
    }


def build_overrides(config: dict, config_run_name: str) -> list[str]:
    w_tertiary = float(config["w_tertiary"])
    sim_thresh = float(config["similarity_threshold"])
    aqe_k = int(config["aqe_k"])
    fic_reg = float(config["fic_regularisation"])
    return [
        f"project.run_name={config_run_name}",
        f"project.output_dir={DATA_OUT}",
        f"stage0.input_dir={DATA_RAW}",
        "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
        f"stage4.association.query_expansion.k={aqe_k}",
        "stage4.association.query_expansion.alpha=5.0",
        "stage4.association.query_expansion.dba=false",
        f"stage4.association.graph.similarity_threshold={sim_thresh}",
        f"stage4.association.solver={SOLVER}",
        f"stage4.association.graph.algorithm={ALGORITHM}",
        f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
        f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
        f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
        f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
        f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
        f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
        "stage4.association.mutual_nn.top_k_per_query=20",
        "stage4.association.fic.enabled=true",
        f"stage4.association.fic.regularisation={fic_reg}",
        f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
        f"stage4.association.reranking.k1={RERANKING_K1}",
        f"stage4.association.reranking.k2={RERANKING_K2}",
        f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
        f"stage4.association.camera_pair_norm.enabled={str(CAMERA_PAIR_NORM).lower()}",
        "stage4.association.fac.enabled=false",
        f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
        f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
        f"stage4.association.multi_query.dir={RUN_DIR / 'stage2'}",
        f"stage4.association.aflink.enabled={str(AFLINK_ENABLED).lower()}",
        f"stage4.association.aflink.max_time_gap_frames={AFLINK_MAX_TIME_GAP_FRAMES}",
        f"stage4.association.aflink.max_spatial_gap_px={AFLINK_MAX_SPATIAL_GAP_PX}",
        f"stage4.association.aflink.min_direction_cos={AFLINK_MIN_DIRECTION_COS}",
        f"stage4.association.aflink.min_velocity_ratio={AFLINK_MIN_VELOCITY_RATIO}",
        f"stage4.association.aflink.velocity_window={AFLINK_VELOCITY_WINDOW}",
        "stage4.association.secondary_embeddings.path=",
        "stage4.association.secondary_embeddings.weight=0.0",
        f"stage4.association.tertiary_embeddings.path={TERTIARY_DINOV2_PATH}",
        f"stage4.association.tertiary_embeddings.weight={w_tertiary}",
        f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
        f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
        f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
        "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
        f"stage4.association.zone_model.bonus={ZONE_BONUS}",
        f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
        f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
        f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
        f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
        f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
        "stage4.association.hierarchical.max_merge_size=12",
        f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
        f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
        f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
        "stage4.association.gallery_expansion.enabled=true",
        f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
        f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
        "stage4.association.weights.length_weight_power=0.3",
        "stage4.association.temporal_overlap.enabled=true",
        "stage4.association.temporal_overlap.bonus=0.05",
        "stage4.association.temporal_overlap.max_mean_time=5.0",
        f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
        "stage5.stationary_filter.enabled=true",
        "stage5.stationary_filter.min_displacement_px=150",
        "stage5.stationary_filter.max_mean_velocity_px=2.0",
        "stage5.min_submission_confidence=0.15",
        "stage5.cross_id_nms_iou=0.40",
        "stage5.min_trajectory_confidence=0.30",
        "stage5.min_trajectory_frames=40",
        "stage5.track_edge_trim.enabled=false",
        "stage5.track_smoothing.enabled=false",
        "stage5.gt_frame_clip=true",
        "stage5.gt_zone_filter=true",
        f"stage5.ground_truth_dir={GT_DIR}",
    ]


def make_features_for_embedding(embedding_path: Path):
    embedding_matrix = np.load(embedding_path).astype(np.float32)
    if embedding_matrix.shape != ORIGINAL_PRIMARY_EMBEDDINGS.shape:
        raise ValueError(f"{embedding_path} shape {embedding_matrix.shape} != {ORIGINAL_PRIMARY_EMBEDDINGS.shape}")
    return [replace(feature, embedding=embedding_matrix[index]) for index, feature in enumerate(ORIGINAL_FEATURES)]


def run_config(config: dict, embedding_path: Path, repool_info: dict | None = None) -> dict:
    label = config["label"]
    mode = config["aggregation_mode"]
    config_run_name = f"{RUN_NAME}_{label}_{mode}"
    config_dir = RUN_DIR / label
    config_dir.mkdir(parents=True, exist_ok=True)
    aqe_k = int(config["aqe_k"])
    fic_reg = float(config["fic_regularisation"])
    run_features = make_features_for_embedding(embedding_path)
    print("\n" + "=" * 80)
    print(
        f"Running {label}/{mode}: w_tertiary={config['w_tertiary']:.3f}, "
        f"similarity_threshold={config['similarity_threshold']:.2f}, aqe_k={aqe_k}, fic_reg={fic_reg:.2f}"
    )
    print("=" * 80)

    cfg = load_config("configs/default.yaml", dataset_config="configs/datasets/cityflowv2.yaml", overrides=build_overrides(config, config_run_name))
    save_config(cfg, config_dir / "config.yaml")

    start = time.time()
    faiss_index, metadata_store = run_stage3(cfg, run_features, tracklets_by_camera, output_dir=config_dir / "stage3")
    stage3_min = (time.time() - start) / 60.0
    print(f"{label} Stage 3 complete in {stage3_min:.2f} min")

    start = time.time()
    trajectories = run_stage4(cfg, faiss_index, metadata_store, run_features, tracklets_by_camera, output_dir=config_dir / "stage4")
    stage4_min = (time.time() - start) / 60.0
    print(f"{label} Stage 4 complete in {stage4_min:.2f} min: {len(trajectories)} global trajectories")

    start = time.time()
    run_stage5(cfg, trajectories, output_dir=config_dir / "stage5")
    stage5_min = (time.time() - start) / 60.0
    print(f"{label} Stage 5 complete in {stage5_min:.2f} min")

    report_path = config_dir / "stage5" / "evaluation_report.json"
    metrics = load_metrics(report_path)
    idf1_value = metrics.get("mtmc_idf1")
    if idf1_value is None:
        idf1_value = metrics.get("trackeval_idf1")
    if idf1_value is None:
        raise RuntimeError(f"IDF1 not found in {report_path}")

    row = {
        "label": label,
        "block": config["block"],
        "aggregation_mode": mode,
        "w_tertiary": float(config["w_tertiary"]),
        "w_primary": round(1.0 - float(config["w_tertiary"]), 6),
        "w_secondary": 0.0,
        "similarity_threshold": float(config["similarity_threshold"]),
        "aqe_k": aqe_k,
        "fic_regularisation": fic_reg,
        "min_k": MIN_K,
        "fallback_count": int((repool_info or {}).get("fallback_count", 0)),
        "notes": config["notes"],
        "idf1": idf1_value,
        "mtmc_idf1": metrics.get("mtmc_idf1"),
        "trackeval_idf1": metrics.get("trackeval_idf1"),
        "mota": metrics.get("mota"),
        "hota": metrics.get("hota"),
        "id_switches": metrics.get("id_switches"),
        "conflations": metrics.get("conflations"),
        "fragmentations": metrics.get("fragmentations"),
        "num_pred_ids": metrics.get("num_pred_ids"),
        "stage_timings_min": {
            "stage3": round(stage3_min, 2),
            "stage4": round(stage4_min, 2),
            "stage5": round(stage5_min, 2),
        },
        "paths": {
            "embedding_path": str(embedding_path),
            "config_dir": str(config_dir),
            "evaluation_report": str(report_path),
            "repool_summary": (repool_info or {}).get("summary_path"),
        },
        "repool_info": repool_info or {},
    }
    print(f"{label}/{mode} MTMC IDF1: {idf1_value:.5f}")
    return row


results = []
aggregation_results = []
halt_reason = None
drift_detected = False
drift_check_result = None
wall_start = time.time()

m0_config = SWEEP_CONFIGS[0]
drift_check_result = run_config(m0_config, PRIMARY_EMBEDDINGS_PATH)
results.append(drift_check_result)
(RUN_DIR / "14h_partial_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

if abs(float(drift_check_result["idf1"]) - M0_REPRO_TARGET) > M0_REPRO_TOL:
    drift_detected = True
    halt_reason = (
        f"M0 drift gate failed: got {drift_check_result['idf1']:.5f}, "
        f"expected within {M0_REPRO_TARGET:.5f} +/- {M0_REPRO_TOL:.5f}"
    )
    print(halt_reason)
else:
    print(f"M0 drift gate passed: {drift_check_result['idf1']:.5f}")
    for config in SWEEP_CONFIGS[1:]:
        mode = config["aggregation_mode"]
        repool_summary_path = RUN_DIR / config["label"] / f"repool_{mode}.json"
        embedding_path = RUN_DIR / "stage2" / f"embeddings_{mode}.npy"
        repool_info = repool_stage2(
            stage2_dir=RUN_DIR / "stage2",
            mode=mode,
            min_k=MIN_K,
            output=embedding_path,
            summary=repool_summary_path,
        )
        repool_info["summary_path"] = str(repool_summary_path)
        row = run_config(config, embedding_path, repool_info=repool_info)
        results.append(row)
        aggregation_results.append(row)
        (RUN_DIR / "14h_partial_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

overall_best = max(results, key=lambda row: row["idf1"] if row["idf1"] is not None else -1.0)
best = overall_best

if drift_detected:
    verdict = "DRIFT"
elif float(overall_best["idf1"]) >= WIN_THRESHOLD:
    verdict = "WIN"
elif float(overall_best["idf1"]) >= MARGINAL_MIN:
    verdict = "MARGINAL"
elif float(overall_best["idf1"]) >= NEUTRAL_MIN:
    verdict = "NEUTRAL"
else:
    verdict = "DEAD"

promoted_config = overall_best if verdict == "WIN" and not drift_detected else None

print("\n" + "#" * 80)
print(
    f"BEST 14h CONFIG: {overall_best['label']} mode={overall_best['aggregation_mode']} "
    f"w_tertiary={overall_best['w_tertiary']:.3f} sim_thresh={overall_best['similarity_threshold']:.2f} "
    f"aqe_k={overall_best['aqe_k']} fic_reg={overall_best['fic_regularisation']:.2f} "
    f"MTMC_IDF1={overall_best['idf1']:.5f} verdict={verdict} drift={drift_detected}"
)
if halt_reason:
    print(f"HALT: {halt_reason}")
print("#" * 80)

## 7. Save 14h Summary

In [ ]:
import json, tarfile, time
from pathlib import Path

import numpy as np

checkpoint_path = Path("/kaggle/working/checkpoint.tar.gz")
metadata_path = Path("/kaggle/working/run_metadata.json")
summary_dir = Path("/kaggle/working/outputs/14h_v1_summary")
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / "14h_summary.json"
root_summary_path = Path("/kaggle/working/14h_summary.json")

primary_embeddings = np.load(PRIMARY_EMBEDDINGS_PATH)
tertiary_embeddings = np.load(TERTIARY_DINOV2_PATH)
with np.load(MULTI_QUERY_PATH) as mq_data:
    multi_query_shape = list(mq_data["embeddings"].shape)

summary = {
    "run_name": RUN_NAME,
    "source_10a_run_name": PREV_RUN_NAME,
    "experiment": "14h-robust-tracklet-pooling",
    "variant": TTA_RECIPE["variant"],
    "kernel": "yahiaakhalafallah/14h-robust-tracklet-pooling",
    "tta_recipe": TTA_RECIPE,
    "aggregation_modes": AGGREGATION_MODES,
    "frame_id_convention": "0-based internal Stage 1/2 frame IDs; MOT output is converted to 1-based in Stage 5",
    "drift_detected": drift_detected,
    "drift_check_label": "M0",
    "drift_check_result": drift_check_result,
    "drift_reproduction_target": M0_REPRO_TARGET,
    "drift_reproduction_tolerance": M0_REPRO_TOL,
    "promoted_config": promoted_config,
    "verdict": verdict,
    "mini_sweep": {
        "description": "Existing Stage-2 pool drift check plus robust aggregation modes at the 14e B1 anchor",
        "grid": SWEEP_CONFIGS,
        "execution_order": [row["label"] for row in results],
        "results": results,
        "best": overall_best,
        "grid_size": len(SWEEP_CONFIGS),
        "halt_reason": halt_reason,
    },
    "results": results,
    "aggregation_results": aggregation_results,
    "overall_best": overall_best,
    "best": overall_best,
    "halt_reason": halt_reason,
    "total_config_count": len(SWEEP_CONFIGS),
    "executed_config_count": len(results),
    "feature_sources": {
        "stage1_tracklets": str(PREV_RUN_DIR / "stage1"),
        "stage2_features": str(RUN_DIR / "stage2"),
        "primary_embeddings": str(PRIMARY_EMBEDDINGS_PATH),
        "tertiary_embeddings": str(TERTIARY_DINOV2_PATH),
        "multi_query_embeddings": str(MULTI_QUERY_PATH),
        "num_features": len(features),
        "primary_shape": list(primary_embeddings.shape),
        "tertiary_shape": list(tertiary_embeddings.shape),
        "multi_query_shape": multi_query_shape,
    },
    "fixed_config": {
        "pca_components": 384,
        "algorithm": ALGORITHM,
        "gallery_expansion": True,
        "gallery_threshold": GALLERY_THRESH,
        "orphan_match_threshold": ORPHAN_MATCH_THRESH,
        "temporal_overlap_bonus": 0.05,
        "intra_merge": INTRA_MERGE,
        "intra_merge_threshold": INTRA_MERGE_THRESH,
        "intra_merge_gap": INTRA_MERGE_GAP,
        "mtmc_only_submission": MTMC_ONLY,
        "w_secondary": 0.0,
        "multi_query_k": 24,
        "robust_pool_min_k": MIN_K,
    },
    "stop_criteria": {
        "win_threshold": WIN_THRESHOLD,
        "marginal_min": MARGINAL_MIN,
        "neutral_min": NEUTRAL_MIN,
        "dead_below": NEUTRAL_MIN,
        "drift_condition": f"abs(M0 - {M0_REPRO_TARGET}) > {M0_REPRO_TOL}",
    },
    "stage_timings_min": {
        "stage2": round(elapsed / 60.0, 2),
        "stage345_sweep": round((time.time() - wall_start) / 60.0, 2),
    },
    "paths": {
        "run_dir": str(RUN_DIR),
        "summary": str(summary_path),
        "root_summary": str(root_summary_path),
    },
}

metadata_path.write_text(json.dumps({"run_name": RUN_NAME}, indent=2), encoding="utf-8")
summary_payload = json.dumps(summary, indent=2)
summary_path.write_text(summary_payload, encoding="utf-8")
root_summary_path.write_text(summary_payload, encoding="utf-8")

print(f"Building checkpoint for {RUN_NAME}")
with tarfile.open(str(checkpoint_path), "w:gz") as tar:
    tar.add(str(metadata_path), arcname="run_metadata.json")
    tar.add(str(summary_path), arcname="outputs/14h_v1_summary/14h_summary.json")
    tar.add(str(root_summary_path), arcname="14h_summary.json")
    for stage in ["stage1", "stage2"]:
        stage_dir = RUN_DIR / stage
        count = 0
        if stage_dir.exists():
            for file_path in stage_dir.rglob("*"):
                if file_path.is_file():
                    tar.add(str(file_path), arcname=f"{RUN_NAME}/{stage}/{file_path.relative_to(stage_dir)}")
                    count += 1
        print(f"  added {stage}: {count} files")
    for file_path in RUN_DIR.rglob("*"):
        if file_path.is_file() and "stage1" not in file_path.parts and "stage2" not in file_path.parts:
            tar.add(str(file_path), arcname=f"{RUN_NAME}/{file_path.relative_to(RUN_DIR)}")

size_mb = checkpoint_path.stat().st_size / 1024**2
print(f"Saved summary: {summary_path}")
print(f"Checkpoint: {checkpoint_path} ({size_mb:.1f} MB)")
print(json.dumps(summary, indent=2))